In [10]:
import os
os.chdir("/Users/nemethandras/Documents/UNI/Bachelor/scrna_project")

# Confirm
print(os.getcwd())
print(os.listdir("."))


/Users/nemethandras/Documents/UNI/Bachelor/scrna_project
['config', '.snakemake', 'results', 'logs', 'workflow', '.vscode', 'data', 'notebooks']


In [11]:
import subprocess

# 1. How many reads actually aligned to chr22?
result = subprocess.run(
    ["samtools", "flagstat", "results/bam/SRR1258218.sorted.bam"],
    capture_output=True, text=True
)
print(result.stdout)

78496 + 0 in total (QC-passed reads + QC-failed reads)
63142 + 0 primary
15354 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
78496 + 0 mapped (100.00% : N/A)
63142 + 0 primary mapped (100.00% : N/A)
63142 + 0 paired in sequencing
31571 + 0 read1
31571 + 0 read2
63142 + 0 properly paired (100.00% : N/A)
63142 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)



In [12]:
# 2. Peek at the raw VCF — header + first few variants
with open("results/vcf/SRR1258218.raw.vcf") as f:
    for line in f:
        print(line.strip())
        # stop after we've seen a few data lines (non-header)
        if not line.startswith("#") and sum(
            1 for l in open("results/vcf/SRR1258218.raw.vcf")
            if not l.startswith("#")
        ) > 0:
            break

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##bcftoolsVersion=1.23.1+htslib-1.23.1
##bcftoolsCommand=mpileup -f data/reference/chr22.fa -q 20 -Q 20 --output-type b -o results/vcf/SRR1258218.mpileup.bcf results/bam/SRR1258218.sorted.bam
##reference=file://data/reference/chr22.fa
##contig=<ID=chr22,length=50818468>
##ALT=<ID=*,Description="Represents allele(s) other than observed.">
##INFO=<ID=INDEL,Number=0,Type=Flag,Description="Indicates that the variant is an INDEL.">
##INFO=<ID=IDV,Number=1,Type=Integer,Description="Maximum number of raw reads supporting an indel">
##INFO=<ID=IMF,Number=1,Type=Float,Description="Maximum fraction of raw reads supporting an indel">
##INFO=<ID=DP,Number=1,Type=Integer,Description="Raw read depth">
##INFO=<ID=VDB,Number=1,Type=Float,Description="Variant Distance Bias for filtering splice-site artefacts in RNA-seq data (bigger is better)",Version="3">
##INFO=<ID=RPBZ,Number=1,Type=Float,Description="Mann-Whitney U-z test of R

In [13]:
# 3. Count variants at each stage
import subprocess

def count_variants(vcf_path):
    result = subprocess.run(
        f"grep -v '^#' {vcf_path} | wc -l",
        shell=True, capture_output=True, text=True
    )
    return int(result.stdout.strip())

raw      = count_variants("results/vcf/SRR1258218.raw.vcf")
filtered = count_variants("results/vcf/SRR1258218.filtered.vcf")

print(f"Raw variants:      {raw}")
print(f"Filtered variants: {filtered}")
print(f"Removed by filter: {raw - filtered}")

Raw variants:      2855
Filtered variants: 2124
Removed by filter: 731


In [14]:
# 4. Look at the actual variant lines
with open("results/vcf/SRR1258218.filtered.vcf") as f:
    for line in f:
        if not line.startswith("#"):
            fields = line.strip().split("\t")
            print(f"Chr: {fields[0]}  Pos: {fields[1]}  "
                  f"Ref: {fields[3]}  Alt: {fields[4]}  "
                  f"Qual: {fields[5]}  Filter: {fields[6]}")

Chr: chr22  Pos: 10580443  Ref: T  Alt: G  Qual: 62.4147  Filter: PASS
Chr: chr22  Pos: 10580462  Ref: G  Alt: C  Qual: 78.4149  Filter: PASS
Chr: chr22  Pos: 10580463  Ref: A  Alt: G  Qual: 77.4149  Filter: PASS
Chr: chr22  Pos: 10580467  Ref: T  Alt: C  Qual: 78.4149  Filter: PASS
Chr: chr22  Pos: 10580500  Ref: C  Alt: T  Qual: 49.4146  Filter: PASS
Chr: chr22  Pos: 10580550  Ref: A  Alt: G  Qual: 56.4147  Filter: PASS
Chr: chr22  Pos: 10580579  Ref: C  Alt: G  Qual: 131.416  Filter: PASS
Chr: chr22  Pos: 10580585  Ref: A  Alt: G  Qual: 131.416  Filter: PASS
Chr: chr22  Pos: 10580592  Ref: C  Alt: T  Qual: 131.416  Filter: PASS
Chr: chr22  Pos: 10580596  Ref: C  Alt: A  Qual: 131.416  Filter: PASS
Chr: chr22  Pos: 10580604  Ref: G  Alt: C  Qual: 131.416  Filter: PASS
Chr: chr22  Pos: 10580632  Ref: A  Alt: G  Qual: 19.4636  Filter: PASS
Chr: chr22  Pos: 10940658  Ref: A  Alt: C  Qual: 10.7923  Filter: PASS
Chr: chr22  Pos: 10941750  Ref: C  Alt: T  Qual: 27.4222  Filter: PASS
Chr: c

In [15]:
# How many reads aligned at all?
!samtools flagstat results/bam/SRR1258218.sorted.bam

78496 + 0 in total (QC-passed reads + QC-failed reads)
63142 + 0 primary
15354 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
78496 + 0 mapped (100.00% : N/A)
63142 + 0 primary mapped (100.00% : N/A)
63142 + 0 paired in sequencing
31571 + 0 read1
31571 + 0 read2
63142 + 0 properly paired (100.00% : N/A)
63142 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


In [16]:
# How deep is the coverage at any position on chr22?
!samtools depth results/bam/SRR1258218.sorted.bam | sort -k3 -rn | head -10

chr22	22901012	964
chr22	22901011	952
chr22	22901022	951
chr22	22901034	950
chr22	22901035	949
chr22	22901033	949
chr22	22901029	944
chr22	22901009	944
chr22	22901013	942
chr22	22901010	942
sort: Broken pipe


In [17]:
# Does the BAM actually have reads in it?
!samtools view -c results/bam/SRR1258218.sorted.bam

78496
